<a href="https://colab.research.google.com/github/col38470682/GVH-Dynamique/blob/main/GVH_Positive_Hierarchy_Benchmark_0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistique directe primaire de phase

Les coefficients complexes aux deux fréquences injectées sont notés :

\[
Z_L
=
\widehat{\dot\theta_M}(\omega_L),
\qquad
Z_U
=
\widehat{\dot\theta_M}(\omega_U).
\]

La statistique directe primaire est définie par :

\[
T_{\phi}
=
\cos\left(
\arg Z_U-\arg Z_L
\right).
\]

Cette statistique est indépendante des amplitudes spectrales et mesure uniquement l'alignement relatif des phases complexes.

Dans le benchmark positif relationnel :

\[
\phi_U=\phi_L,
\]

donc :

\[
T_{\phi}^{obs}\approx 1.
\]

Sous le null relationnel strict, \(\phi_L\) et \(\phi_U\) sont tirées indépendamment, ce qui détruit la relation de phase tout en préservant les marginales uniformes.

Un succès direct local est défini par :

\[
T_{\phi}^{obs,(k)}
>
Q_{97.5}^{null,(k)}.
\]

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

N = 8192
t_min = 0.0
t_max = 200.0 * np.pi
t = np.linspace(t_min, t_max, N, endpoint=False)
dt = t[1] - t[0]

omega_M = 1.00
omega_L = 0.10
omega_U = 0.03

m_L = 0.30
m_U = 0.20

print("Paramètres FM 0.3 chargés")
print(f"N={N}, dt={dt:.12f}")
print(f"omega_M={omega_M}, omega_L={omega_L}, omega_U={omega_U}")
print(f"m_L={m_L}, m_U={m_U}")

Paramètres FM 0.3 chargés
N=8192, dt=0.076699039394
omega_M=1.0, omega_L=0.1, omega_U=0.03
m_L=0.3, m_U=0.2


In [3]:
# ============================================================
# GVH Positive Hierarchy Benchmark 0.3
# Test direct relationnel Phase-Locked — K = 20
# ============================================================

import numpy as np
from scipy.stats import binom

# ------------------------------------------------------------
# 1. Paramètres figés
# ------------------------------------------------------------

K = 20
N_surr = 1000
alpha_local = 0.025

seed_targets = 20260710
seed_surr = 20260711

rng_targets = np.random.default_rng(seed_targets)
rng_surr = np.random.default_rng(seed_surr)

# Vérification des variables héritées du benchmark FM
required_vars = [
    "t",
    "omega_M",
    "omega_L",
    "omega_U",
    "m_L",
    "m_U",
]

missing = [name for name in required_vars if name not in globals()]

if missing:
    raise RuntimeError(
        "Variables absentes du runtime : "
        + ", ".join(missing)
        + "\nRéexécute d'abord la cellule des paramètres FM."
    )

# ------------------------------------------------------------
# 2. Grille fréquentielle et bins cibles
# ------------------------------------------------------------

N = len(t)
dt = float(t[1] - t[0])

omega_grid = 2.0 * np.pi * np.fft.rfftfreq(N, d=dt)

idx_L = int(np.argmin(np.abs(omega_grid - omega_L)))
idx_U = int(np.argmin(np.abs(omega_grid - omega_U)))

print("Bins fréquentiels utilisés")
print("=" * 80)
print(
    f"omega_L cible = {omega_L:.12f} | "
    f"omega grille = {omega_grid[idx_L]:.12f} | "
    f"idx = {idx_L}"
)
print(
    f"omega_U cible = {omega_U:.12f} | "
    f"omega grille = {omega_grid[idx_U]:.12f} | "
    f"idx = {idx_U}"
)

# ------------------------------------------------------------
# 3. Fonction T_phi préspécifiée
# ------------------------------------------------------------

def compute_T_phi(theta_dot):
    """
    T_phi = cos(arg(Z_U) - arg(Z_L))

    où :
    Z_L = FFT(theta_dot centered)[idx_L]
    Z_U = FFT(theta_dot centered)[idx_U]
    """

    theta_dot_centered = theta_dot - np.mean(theta_dot)
    fft_vals = np.fft.rfft(theta_dot_centered)

    Z_L = fft_vals[idx_L]
    Z_U = fft_vals[idx_U]

    delta_phase = np.angle(
        np.exp(1j * (np.angle(Z_U) - np.angle(Z_L)))
    )

    T_phi = np.cos(delta_phase)

    return float(T_phi), float(delta_phase)


# ------------------------------------------------------------
# 4. Phases positives hors échantillon
#
# H1 relationnelle :
# phi_L^(k) ~ U(0, 2pi)
# phi_U^(k) = phi_L^(k)
# ------------------------------------------------------------

phi_targets = rng_targets.uniform(
    0.0,
    2.0 * np.pi,
    size=K
)

# ------------------------------------------------------------
# 5. Boucle K = 20
# ------------------------------------------------------------

results = []
success_direct = 0

print("\n")
print("Test direct relationnel Phase-Locked")
print("=" * 100)

for k in range(K):

    # --------------------------------------------------------
    # Observé positif relationnel
    # --------------------------------------------------------

    phi_L_obs = phi_targets[k]
    phi_U_obs = phi_targets[k]   # verrouillage exact

    theta_dot_obs = (
        omega_M
        + m_L * omega_L
        * np.cos(omega_L * t + phi_L_obs)
        + m_U * omega_U
        * np.cos(omega_U * t + phi_U_obs)
    )

    T_obs, delta_obs = compute_T_phi(theta_dot_obs)

    # --------------------------------------------------------
    # Null relationnel strict
    #
    # Marginales préservées :
    # phi_L ~ U(0, 2pi)
    # phi_U ~ U(0, 2pi)
    #
    # Dépendance détruite :
    # phi_U indépendant de phi_L
    # --------------------------------------------------------

    T_null = np.empty(N_surr, dtype=float)

    for s in range(N_surr):

        phi_L_s = rng_surr.uniform(0.0, 2.0 * np.pi)
        phi_U_s = rng_surr.uniform(0.0, 2.0 * np.pi)

        theta_dot_s = (
            omega_M
            + m_L * omega_L
            * np.cos(omega_L * t + phi_L_s)
            + m_U * omega_U
            * np.cos(omega_U * t + phi_U_s)
        )

        T_null[s], _ = compute_T_phi(theta_dot_s)

    # --------------------------------------------------------
    # Résumé statistique local
    # --------------------------------------------------------

    q025 = np.quantile(T_null, 0.025)
    q975 = np.quantile(T_null, 0.975)

    percentile = np.mean(T_null < T_obs)

    # p-value empirique unilatérale supérieure
    p_upper = (
        1.0 + np.sum(T_null >= T_obs)
    ) / (
        N_surr + 1.0
    )

    success = T_obs > q975

    if success:
        success_direct += 1

    results.append({
        "k": k + 1,
        "phi_target": phi_L_obs,
        "delta_phi_obs": delta_obs,
        "T_phi_obs": T_obs,
        "null_mean": np.mean(T_null),
        "null_std": np.std(T_null, ddof=1),
        "Q2.5": q025,
        "Q97.5": q975,
        "percentile": percentile,
        "p_upper": p_upper,
        "success": success,
    })

    print(
        f"Réalisation {k+1:02d}/{K} | "
        f"phi={phi_L_obs:.6f} | "
        f"T_phi={T_obs:.9f} | "
        f"pct={percentile:.3f} | "
        f"p_upper={p_upper:.6f} | "
        f"success={success}"
    )


# ------------------------------------------------------------
# 6. Test global binomial exact
# ------------------------------------------------------------

p_global = binom.sf(
    success_direct - 1,
    K,
    alpha_local
)

benchmark_direct_validated = success_direct >= 3


# ------------------------------------------------------------
# 7. Bilan global
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("BILAN GLOBAL — PHÉNOMÈNE DIRECT RELATIONNEL")
print("=" * 100)

print(f"K réalisations                      : {K}")
print(f"N_surr par réalisation              : {N_surr}")
print(f"Succès directs T_phi                : {success_direct}/{K}")
print(f"Seuil global préspécifié            : X >= 3")
print(
    f"Phénomène direct global validé      : "
    f"{benchmark_direct_validated}"
)
print(
    f"P-value binomial exact globale      : "
    f"{p_global:.12e}"
)

print("=" * 100)

if benchmark_direct_validated:
    print(
        "RÈGLE D'ARRÊT FRANCHIE : "
        "le test GVH unique S_mean est autorisé."
    )
else:
    print(
        "RÈGLE D'ARRÊT NON FRANCHIE : "
        "ne pas interpréter S_mean comme validation."
    )

print("=" * 100)

Bins fréquentiels utilisés
omega_L cible = 0.100000000000 | omega grille = 0.100000000000 | idx = 10
omega_U cible = 0.030000000000 | omega grille = 0.030000000000 | idx = 3


Test direct relationnel Phase-Locked
Réalisation 01/20 | phi=3.747138 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 02/20 | phi=5.862575 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 03/20 | phi=4.924919 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 04/20 | phi=0.532586 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 05/20 | phi=1.309938 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 06/20 | phi=1.949212 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 07/20 | phi=3.081362 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=True
Réalisation 08/20 | phi=2.972401 | T_phi=1.000000000 | pct=1.000 | p_upper=0.000999 | success=

In [4]:
import numpy as np
from scipy.stats import binom

# ============================================================
# Définitions formelles des opérateurs GVH
# ============================================================
def compute_D_T(X):
    X = np.asarray(X, dtype=float)
    return np.linalg.norm(np.diff(X, axis=0), axis=1)

def compute_S_T(D):
    D = np.asarray(D, dtype=float)
    return np.abs(np.diff(D))

# ============================================================
# Initialisation et réplication relationnelle 0.3
# ============================================================
rng_targets = np.random.default_rng(20260709)
rng_surr = np.random.default_rng(20260709 + 1)

K = 20
N_surr = 1000
alpha_local = 0.025

# Génération des 20 paires de phases cibles hors échantillon
phases_L_targets = rng_targets.uniform(0.0, 2.0 * np.pi, size=K)
phases_U_targets = phases_L_targets  # Contrainte structurelle \phi_U = \phi_L

success_gvh_S_mean = 0
results_gvh = []

print("Lancement du test GVH en aveugle unique (S_mean)...")
print("=" * 84)

for k in range(K):
    phi_L_target = phases_L_targets[k]
    phi_U_target = phases_U_targets[k]

    # 1. Génération de l'observé relationnel k
    theta_M_k = (
        omega_M * t
        + m_L * np.sin(omega_L * t + phi_L_target)
        + m_U * np.sin(omega_U * t + phi_U_target)
    )

    X_M_k = np.column_stack([np.cos(theta_M_k), np.sin(theta_M_k)])
    D_k = compute_D_T(X_M_k)
    S_mean_k_obs = np.mean(compute_S_T(D_k))

    # 2. Échantillonnage du Null Relationnel (1000 surrogates décorrélés)
    S_mean_k_null = np.empty(N_surr)

    for s in range(N_surr):
        phi_L_s = rng_surr.uniform(0.0, 2.0 * np.pi)
        phi_U_s = rng_surr.uniform(0.0, 2.0 * np.pi)  # Brisure de la relation

        theta_M_s = (
            omega_M * t
            + m_L * np.sin(omega_L * t + phi_L_s)
            + m_U * np.sin(omega_U * t + phi_U_s)
        )

        X_M_s = np.column_stack([np.cos(theta_M_s), np.sin(theta_M_s)])
        D_s = compute_D_T(X_M_s)
        S_mean_k_null[s] = np.mean(compute_S_T(D_s))

    # 3. Diagnostic statistique unilatéral supérieur
    Q975_S_mean = np.percentile(S_mean_k_null, 97.5)
    pct_local = np.mean(S_mean_k_null <= S_mean_k_obs)
    p_upper_local = (1.0 + np.sum(S_mean_k_null >= S_mean_k_obs)) / (N_surr + 1.0)

    is_gvh_positive = S_mean_k_obs > Q975_S_mean
    if is_gvh_positive:
        success_gvh_S_mean += 1

    results_gvh.append({
        "k": k,
        "obs": S_mean_k_obs,
        "pct": pct_local,
        "p_upper": p_upper_local,
        "success": is_gvh_positive
    })

    print(
        f"Réalisation {k+1:02d}/{K} | "
        f"S_mean_obs = {S_mean_k_obs:.6e} | "
        f"pct = {pct_local:.3f} | "
        f"p_upper = {p_upper_local:.6f} | "
        f"success = {is_gvh_positive}"
    )

# ------------------------------------------------------------
# 4. Évaluation du critère de décision binomial exact global
# ------------------------------------------------------------
gvh_validated = success_gvh_S_mean >= 3
p_value_gvh_global = binom.sf(success_gvh_S_mean - 1, K, alpha_local)

print("\n" + "=" * 84)
print("BILAN GLOBAL — TEST GVH RELATIONNEL (0.3)")
print("=" * 84)
print(f"Nombre de succès locaux (S_mean > Q97.5) : {success_gvh_S_mean}/{K}")
print(f"Seuil minimal préspécifié de validation : X >= 3")
print(f"P-value binomiale exacte globale         : {p_value_gvh_global:.6e}")
print("-" * 84)
print(f"STATUT DU BENCHMARK HIERARCHIQUE GVH     : {'SUCCÈS' if gvh_validated else 'ÉCHEC'}")
print("=" * 84)


Lancement du test GVH en aveugle unique (S_mean)...
Réalisation 01/20 | S_mean_obs = 1.123577e-05 | pct = 0.240 | p_upper = 0.760240 | success = False
Réalisation 02/20 | S_mean_obs = 1.123718e-05 | pct = 0.590 | p_upper = 0.410589 | success = False
Réalisation 03/20 | S_mean_obs = 1.123694e-05 | pct = 0.538 | p_upper = 0.462537 | success = False
Réalisation 04/20 | S_mean_obs = 1.123926e-05 | pct = 0.926 | p_upper = 0.074925 | success = False
Réalisation 05/20 | S_mean_obs = 1.123530e-05 | pct = 0.047 | p_upper = 0.953047 | success = False
Réalisation 06/20 | S_mean_obs = 1.123929e-05 | pct = 0.930 | p_upper = 0.070929 | success = False
Réalisation 07/20 | S_mean_obs = 1.123729e-05 | pct = 0.603 | p_upper = 0.397602 | success = False
Réalisation 08/20 | S_mean_obs = 1.123670e-05 | pct = 0.500 | p_upper = 0.500500 | success = False
Réalisation 09/20 | S_mean_obs = 1.123872e-05 | pct = 0.862 | p_upper = 0.138861 | success = False
Réalisation 10/20 | S_mean_obs = 1.123552e-05 | pct = 0.1

## Conclusion — Benchmark relationnel 0.3 (Phase-Locked)

Le benchmark 0.3 a testé une contrainte relationnelle positive préspécifiée :

\[
\phi_U = \phi_L,
\qquad
\Delta\phi_{LU}=0,
\]

avec des phases absolues variant entre les réalisations.

### 1. Validation directe indépendante de GVH

La statistique spectrale relationnelle préspécifiée

\[
T_\phi
=
\cos\left(
\arg Z_U-\arg Z_L
\right)
\]

détecte systématiquement la contrainte injectée :

\[
20/20
\]

succès locaux, avec

\[
p_{\rm global}
=
9.09\times10^{-33}.
\]

Le phénomène relationnel est donc présent et statistiquement séparable du null relationnel strict.

### 2. Test GVH en aveugle

L'unique observable GVH préspécifiée,

\[
S_{\rm mean}
=
\operatorname{mean}
\left(
|\Delta D_T|
\right),
\]

échoue à rejeter le même null relationnel :

\[
0/20
\]

succès locaux, avec

\[
p_{\rm global}=1.
\]

### 3. Verdict

Le phénomène relationnel de phase est directement détectable, mais il n'est pas détecté par \(S_{\rm mean}\) dans ce benchmark.

Ainsi, le résultat établit une limite empirique précise :

\[
\boxed{
S_{\rm mean}
\text{ ne sépare pas ici le verrouillage }
\phi_U=\phi_L
\text{ du null à phases indépendantes.}
}
\]

Cette conclusion concerne uniquement l'observable \(S_{\rm mean}\), le générateur FM considéré, les paramètres préspécifiés et le protocole statistique utilisé.

Aucune autre observable GVH n'est testée post hoc dans ce notebook.

Le benchmark 0.3 est archivé comme un contrôle négatif préspécifié après validation positive indépendante du phénomène direct.